In [ ]:
%cd ../.
import os, sys
sys.path.insert(0, os.path.abspath('../Scripts'))
sys.path.insert(0, os.path.expanduser('~/CDD_Vault_API/python'))  # CDD Vault API (get_df)

In [ ]:
%load_ext autoreload
%autoreload 2

import re
import os
import pandas as pd
import numpy as np
# import py3Dmol
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
import csv
import pickle
import json

from pathlib import Path
from rdkit import Chem, DataStructs
from rdkit.Chem import AllChem,rdFMCS
# import prolif as plf
from glob import glob
# import meeko
import subprocess as sub
# from vina import Vina
import time
from tqdm import tqdm
tqdm.pandas()
import importlib
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor, HistGradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.dummy import DummyClassifier
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.model_selection import StratifiedKFold, cross_val_predict, KFold, StratifiedGroupKFold
from sklearn import metrics as _m
from sklearn.metrics import f1_score, silhouette_score, davies_bouldin_score, calinski_harabasz_score
from sklearn.metrics import roc_auc_score, average_precision_score, matthews_corrcoef
# from xgboost import XGBClassifier,XGBRegressor
from openTSNE import TSNE as oTSNE        # pip install openTSNE
import seaborn as sns
from scipy import stats
import csv, contextlib, threading, joblib
import joblib
from joblib import Parallel, delayed
from datetime import date
## load internal (in-house) + public-augmented data, champion model, eval helpers
from copy import deepcopy
from itertools import combinations


# user defined modules
import Rdkit_tools as rdkit_tools
importlib.reload(rdkit_tools)
import Molecule as M
import ML_Reg as ML_Reg
import ML_Class as ML_Class
# from MolViz3D import MolViz3D
import Statistics_tools as stats_tools
import python.functions as fn
from python.harmonize_sol import harmonize_solubility

from get_library import get_df   # CDD Vault collection export
from get_protocol_data import get_data, load_config, alias_map
# from tdc.multi_pred import DTI

import yaml
from python.ADME_build_ML import PARAMS, DATA, OUTPUT   # CLI classes reused here (%autoreload picks up edits)
from IPython.display import HTML, display

# FUNCTIONS

# MAIN

## 0. Imports

In [ ]:
## params — every YAML key becomes an attribute (params.ADME_ENDPOINTS); see python/ADME_build_ML.py:PARAMS
CONFIG = 'config/config.yaml'
params = PARAMS(CONFIG)
params.load_params()
## transition shim — keep the bare YAML names (ADME_ENDPOINTS, CUTOFFS, SERAC_C, RF_SINGLETASK, ...) working below
globals().update({k: v for k, v in params.__dict__.items() if k.isupper()})
config = params

In [ ]:
## download library: collapse the per-measurement rows to one row per compound
UB2 = (pd.read_csv('data/FP/20260914_U2.csv')
       .rename(columns={'Molecule Name': 'compound', 'SMILES': 'smiles',
                        'UBR2 standard FP assay: Geomean IC50 (uM)': 'ic50_raw_UB2'})
       [['compound', 'smiles', 'ic50_raw_UB2']]
       .groupby('compound', as_index=False).first())
print(f'> UB2: {len(UB2)} compounds')

## download library: collapse the per-measurement rows to one row per compound
UB1 = (pd.read_csv('data/FP/20260914_U1.csv')
       .rename(columns={'Molecule Name': 'compound', 'SMILES': 'smiles',
                        'IC50': 'ic50_raw_UB1'})
       [['compound', 'smiles', 'ic50_raw_UB1']]
       .groupby('compound', as_index=False).first())
print(f'> UB1: {len(UB1)} compounds')

UB = pd.merge(UB2,UB1,how='left')

## 1. Regression

In [ ]:
## label = geomean IC50 in uM; keep the censored bound as its numeric value, then clip at the cap
IC50_CAP = 100.0                       # uM ceiling for the label; move to config once settled
UB_N     = '_UB2'
UB['label'] = pd.to_numeric(UB['ic50_raw'+UB_N].str.strip().str.lstrip('<>'),
                             errors='coerce').clip(upper=IC50_CAP)

## H237 features = H236 fingerprints/physchem/MACCS/AtomPair + ~200 DS_ descriptors
H237_UB = rdkit_tools.compute_H237_features(UB[['compound', 'smiles']], n_jobs=32, v=True)

## ML frame: the inner merge drops any compound whose SMILES failed to parse
ML_UB = H237_UB.merge(UB[['compound', 'label']], on='compound').dropna(subset=['label'])
print(f'> ML frame: {ML_UB.shape[0]} compounds x {ML_UB.shape[1] - 2} features '
      f'({int((ML_UB.label == IC50_CAP).sum())} clipped at the cap)')

## 5-fold RF cross-validation on FIXED compound folds — the fold column is what enables LOFO calibration
## CAUTION: get_validation_sets hardcodes n_splits=5 and ignores its folds argument
assert RF_SINGLETASK['cv_folds'] == 5, 'get_validation_sets always makes 5 folds'
fold_sets_UB = ML_Reg.get_validation_sets(ML_UB['compound'].tolist())
rf_UB, pred_UB = ML_Reg.K_fold_by_defined_IDs(
    ML_UB, ID='compound', ID_sets=fold_sets_UB,
    model=RandomForestRegressor(**RF_SINGLETASK['champion'],
                                random_state=RF_SINGLETASK['seed'], n_jobs=RF_SINGLETASK['n_jobs']),
    col_to_rm=['compound', 'label'], uq=True, v=True)

## metrics: r2det is the selection metric, r2 is the squared Pearson
metrics_UB = ML_Reg.get_reg_metrics_from_preddf(pred_UB, ntrain=len(ML_UB))

In [ ]:
## showing confidence vs residuals
## calibration on the CV arm — the same recipe the ADME endpoints deploy
calib_UB = ML_Reg.calibrate_confidence_params(pred_UB)
print(f"> rmse_cv = {calib_UB['rmse_cv']:.2f} uM | label_std = {calib_UB['label_std']:.2f} uM | "
      f"|resid| ~ {calib_UB['recal_a']:.2f} + {calib_UB['recal_b']:.2f} * uq_std")

## leave-one-fold-out confidences: each fold is scored by a fit on the other 4, so nothing is in-sample
conf_UB = ML_Reg.apply_confidences_lofo(pred_UB)
conf_UB['residuals'] = (conf_UB['pred_y'] - conf_UB['real_y']).abs()

CONF_COL = 'conf_recal'     # 'conf_labelstd' | 'conf_rmse' | 'conf_conformal'
CSPLIT = params.webapp.get('confidence_split', 0.5)   # same split the webapp colours on
d = conf_UB.dropna(subset=[CONF_COL, 'residuals'])

fig, ax = plt.subplots(figsize=(6.5, 5))
## rank association: does a higher confidence go with a smaller error?
rho, pval = stats.spearmanr(d[CONF_COL], d['residuals'])
ax.scatter(d[CONF_COL], d['residuals'], s=16, alpha=.55, lw=.4,
           color=SERAC_C['azure'], edgecolor='white')
## binned mean |residual| — the trend the eye should follow
g = d['residuals'].groupby(pd.cut(d[CONF_COL], np.linspace(0, 1, 6)), observed=True).mean()
ax.plot([iv.mid for iv in g.index], g.values, '-o', ms=4, lw=1.6, color=SERAC_C['ember'])
## split at the webapp's confidence cut and score each half: a working confidence puts the
## bigger RMSE on the LEFT (least confident) and the smaller one on the RIGHT
ax.axvline(CSPLIT, color='#6f675c', ls='--', lw=1.1, zorder=1)
for half, x, ha in ((d[CONF_COL] < CSPLIT, .03, 'left'), (d[CONF_COL] >= CSPLIT, .97, 'right')):
    res = d['residuals'][half]
    txt = f'RMSE {np.sqrt(np.mean(res ** 2)):.3f}\nn={len(res)}' if len(res) else 'no points'
    ax.text(x, .97, txt, transform=ax.transAxes, ha=ha, va='top', fontsize=9,
            bbox=dict(fc='white', ec='#e3ddd0', boxstyle='round,pad=0.3', alpha=.9))
ax.set_title(f'UBR2 FP IC50 — rho={rho:.2f} (p={pval:.2g}), n={len(d)}', fontsize=10)
ax.set_xlabel(f'RF confidence ({CONF_COL})'); ax.set_ylabel('|residual| (uM)')
ax.set_xlim(0, 1); ax.grid(alpha=.25)
fig.suptitle(f'augmented_cv — {CONF_COL} vs |residual|, split at {CSPLIT} (held-out folds)', y=.99)
fig.tight_layout()
plt.show()

## 2. Classification

In [ ]:
## hit call on the chosen target — a separate column, so the regression label above survives
HIT_UM = 1.0                          # uM; a compound is a hit below this IC50
UB_N   = '_UB2'                        # '_UB1' | '_UB2' — same switch as the regression section
_raw   = UB['ic50_raw' + UB_N].str.strip()
_ic50  = pd.to_numeric(_raw.str.lstrip('<>'), errors='coerce')
## unmeasured must stay NA, and a '>' bound below the cutoff cannot be called either way
UB['hit'] = (_ic50 < HIT_UM).astype('boolean').mask(
    _ic50.isna() | (_raw.str.startswith('>').fillna(False) & (_ic50 < HIT_UM)))
print(f'> {UB_N[1:]} at {HIT_UM:g} uM: hits {int((UB.hit == True).sum())} | '
      f'non-hits {int((UB.hit == False).sum())} | unusable {int(UB.hit.isna().sum())} of {len(UB)}')

## ML frame: reuse the H237 block from the regression section, swap in the hit call as the label
ML_UB_clf = (H237_UB.merge(UB[['compound', 'hit']], on='compound').dropna(subset=['hit'])
             .rename(columns={'hit': 'label'}).astype({'label': int}))
print(f'> ML frame: {ML_UB_clf.shape[0]} compounds x {ML_UB_clf.shape[1] - 2} features '
      f'({ML_UB_clf.label.mean():.1%} hits)')
## the rare class must reach one member per fold, or StratifiedGroupKFold cannot split
## CAUTION: this only catches the hard failure — read the printed counts to judge if n is usable
assert ML_UB_clf.label.value_counts().min() >= RF_SINGLETASK['cv_folds'], \
    f'rare class has {ML_UB_clf.label.value_counts().min()} compounds, below {RF_SINGLETASK["cv_folds"]} folds'

## folds grouped by skeleton InChIKey so salts/stereoisomers/tautomers never split, and stratified
## on the hit call to hold the class ratio (config FOLD_GROUP_BY_INCHIKEY)
_grp = ML_UB_clf['compound'].map(dict(zip(UB['compound'], fn.inchikeys_for(UB, level='skeleton'))))
_ids = ML_UB_clf['compound'].to_numpy()
folds_clf = [[list(_ids[tr]), list(_ids[te])] for tr, te in
             StratifiedGroupKFold(n_splits=RF_SINGLETASK['cv_folds'], shuffle=True,
                                  random_state=RF_SINGLETASK['seed']
                                  ).split(_ids, ML_UB_clf['label'], _grp)]

## 5-fold CV with ONE decision tree; n_estimators and n_jobs are RF-only, so drop them
## flip back to the forest: RandomForestClassifier(**RF_SINGLETASK['champion'],
##     random_state=RF_SINGLETASK['seed'], n_jobs=RF_SINGLETASK['n_jobs'])
TREE_KW = {k: v for k, v in RF_SINGLETASK['champion'].items() if k != 'n_estimators'}
clf_UB, pred_clf = ML_Class.K_fold_by_defined_IDs_Classification(
    ML_UB_clf, ID='compound', ID_sets=folds_clf,
    model=DecisionTreeClassifier(**TREE_KW, random_state=RF_SINGLETASK['seed']),
    col_to_rm=['compound', 'label'], v=True)

## ROC curve — CAUTION: one tree emits only a handful of distinct probabilities, so the curve
## is a coarse staircase and its ROC-AUC understates the model against a forest
roc_UB = ML_Class.plot_roc_curve([pred_clf], c=[SERAC_C['azure']],
                                 l=[f'{UB_N[1:]} — hit < {HIT_UM:g} uM'],
                                 metric2show=['roc_auc', 'pr_auc', 'MCC'])
plt.title(f"{UB_N[1:]} FP hit classifier — one tree, {RF_SINGLETASK['cv_folds']}-fold CV, skeleton-InChIKey folds")
plt.show()

## roc_UB maps every probability threshold to its fpr/tpr — read an operating point off it
print(pd.crosstab(pred_clf['real_y'], pred_clf['pred_y'], rownames=['actual'], colnames=['predicted']))

## 3. Decision tree visualization

In [ ]:
## unhashed bits: one column per atom environment, so a split names ONE substructure.
## The H237 block in section 2 is HASHED — collisions make a split untraceable.
MF_RADIUS, MF_MIN_CPDS = 2, 2          # radius, and the least compounds a bit must appear in
X_UB, bits_UB = rdkit_tools.unhashed_morgan(UB, radius=MF_RADIUS, min_compounds=MF_MIN_CPDS,
                                            return_bits=True, v=True)
print(bits_UB[bits_UB.kept].radius.value_counts().sort_index().rename('kept bits').to_frame().T.to_string())

## same hit call as section 2, now on the unhashed bits instead of H237
ML_UB_bits = (X_UB.merge(UB[['compound', 'hit']], on='compound').dropna(subset=['hit'])
              .rename(columns={'hit': 'label'}).astype({'label': int}))
FEAT_UB = [c for c in ML_UB_bits.columns if c not in ('compound', 'label')]
print(f'> ML frame: {ML_UB_bits.shape[0]} compounds x {len(FEAT_UB)} bits '
      f'({ML_UB_bits.label.mean():.1%} hits)')

## reuse section 2's folds, so the bits model is comparable with the H237 one
assert set(ML_UB_bits.compound) == set(ML_UB_clf.compound), 'compound sets differ — rebuild the folds'

## depth 3 keeps the whole tree on one readable figure
TREE_DEPTH = 3
_tree_kw = dict(max_depth=TREE_DEPTH, min_samples_leaf=20, random_state=RF_SINGLETASK['seed'])
_, pred_bits = ML_Class.K_fold_by_defined_IDs_Classification(
    ML_UB_bits, ID='compound', ID_sets=folds_clf,
    model=DecisionTreeClassifier(**_tree_kw), col_to_rm=['compound', 'label'], v=True)
metrics_bits = ML_Class.metrics_from_pred_df(pred_bits)
print(f"> depth-{TREE_DEPTH} tree on unhashed bits: ROC-AUC {metrics_bits['roc_auc']:.3f} | "
      f"PR-AUC {metrics_bits['pr_auc']:.3f} | MCC {matthews_corrcoef(pred_bits.real_y, pred_bits.pred_y):.3f} | "
      f"F1 {f1_score(pred_bits.real_y, pred_bits.pred_y):.3f}")
print(pd.crosstab(pred_bits['real_y'], pred_bits['pred_y'], rownames=['actual'], colnames=['predicted']))

## one tree on every row for the picture — the CV runner returns only the last fold's model
tree_UB = DecisionTreeClassifier(**_tree_kw).fit(ML_UB_bits[FEAT_UB], ML_UB_bits['label'])

## structures for the split boxes — UB carries smiles, not mol objects, so parse them here
MOL_UB = dict(zip(UB['compound'], UB['smiles'].map(Chem.MolFromSmiles)))

## CAUTION: draw_bits=True puts INTERNAL structures into the image. Keep this figure on this
## machine — do not paste it into a slide, ticket or any tool that uploads. Pass mols=None (or
## draw_bits=False) to get the structure-free version, where boxes carry the bit id only.
fig = rdkit_tools.plot_decision_tree_MF_bits(tree_UB, ML_UB_bits, mols=MOL_UB, draw_bits=True,
                                             radius=MF_RADIUS,
                                             pos_label=f'{UB_N[1:]} hit < {HIT_UM:g} uM')
plt.show()

## 4. Addressing UB selectivity

In [ ]:
UB_sel = UB[UB['ic50_raw_UB1'].notnull() & UB['ic50_raw_UB2'].notnull()].copy()
UB_sel['ic50_raw_UB2'] = pd.to_numeric(UB_sel['ic50_raw_UB2'].str.strip().str.lstrip('<>'),errors='coerce')
UB_sel['ic50_raw_UB1'] = pd.to_numeric(UB_sel['ic50_raw_UB1'].str.strip().str.lstrip('<>'),errors='coerce')#.clip(upper=100)
sns.scatterplot(x='ic50_raw_UB1',y='ic50_raw_UB2',data=UB_sel)